<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.5rem; margin-bottom: 0.2rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.4rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfire 
</h2>
  <hr class="title-rule> style=<margin-bottom: 5rem;">
</div>

## Module 5: *Subset and Split*
##### Version Number: 4.0
---
### Contents  
> 1. *Filter Dates for modeling*
> 2. *Split and Scale Data*
> 3. *Export Data*
---
### Notes
---
### Inputs
- `X`,`y`,`details` - Modeling sets
---
### Outputs 
- `X_scaled`,`y_reduced`,`details_reduced` - Scaled Modeling sets (sometimes with reduced modeling sets to preserve processor)
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to create all second degree interactions possible in a dataset, then name these terms
# in the format of feature1_x_feature2. Returns dataframe of interactions.
from src.data_utils import create_2nd_degree_interactions
from src.data_utils import create_interactions

# A space saving function to rank interactions
from src.data_utils import rank_variables_by_correlation

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling prep
from sklearn.preprocessing import MinMaxScaler

---

###  Load Data

In [3]:
samples = pd.read_csv("../data/processed/variable_selection_output.csv")

# One Hot Encoding

One hot encode appropriate categorical variables.

In [4]:
categorical_columns = ['dominant_province_description','dominant_section_description','Season']
one_hot = samples[categorical_columns]
samples = samples.drop(columns = ['dominant_province_description','dominant_section_description','Season'])

In [5]:
one_hot_samples = pd.get_dummies(
    one_hot,
    columns=categorical_columns,
    drop_first=False
)

In [6]:
coded_samples = pd.concat([samples,one_hot_samples],axis=1)

## Filter Dates for modeling

- **01/01/2018 - 12/31/2024** Dates to train the models
- **01/01/2025 - 01/23/2025** Dates for evaluating model performance on unseen data

In [7]:
coded_samples['Date'] = pd.to_datetime(coded_samples['Date'])

# first day to analyze in weather dataset
FIRST_DATE = pd.to_datetime('2018-01-01').date()

# last day to analyze in weather dataset
LAST_DATE = pd.to_datetime('2024-12-31').date()

# Boolean mask for dates within the range
mask = (coded_samples['Date'].dt.date >= FIRST_DATE) & (coded_samples['Date'].dt.date <= LAST_DATE)

pal = coded_samples.loc[~mask]

# Keep only rows within the range
model_samples = coded_samples.loc[mask].copy()

In [8]:
model_samples['Target'].value_counts()

Target
0    471487
1    107147
2     24818
Name: count, dtype: int64

### (OPTIONAL) Subset classes to save processor
- All classes are downsized for working on models

In [9]:
model_samples['Date'] = pd.to_datetime(model_samples['Date'])

# Number of rows to keep from classes 
n_rows = 50000

# Split by class
class0 = model_samples[model_samples['Target'] == 0]
class1 = model_samples[model_samples['Target'] == 1]
class2 = model_samples[model_samples['Target'] == 2]

# Sample classes down to specified rows
class0_sampled = class0.sample(n=n_rows, random_state=14)
class1_sampled = class1.sample(n=n_rows, random_state=14)
#class2_sampled = class2.sample(n=n_rows, random_state=14)

# Combine classes back together
reduced = pd.concat([class0_sampled, class1_sampled, class2], ignore_index=True)

In [10]:
reduced['Target'].value_counts()

Target
0    50000
1    50000
2    24818
Name: count, dtype: int64

In [11]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date','Target', 'grid_id',
       'geometry', 'fire_count', 'total_fire_damage','acres','area_in_cali',
        'centroid_easting', 'centroid_northing','maximum_x', 'minimum_y',
       'maximum_y', 'minimum_x']

y = reduced['Target']
pal_y = pal['Target']

X = reduced.drop(columns=text_columns)
pal_X = pal.drop(columns=text_columns)

details = reduced[text_columns]
pal_details = pal[text_columns]

### Rank variables by correlation strength

In [12]:
rank_variables_by_correlation(X,y)

,Feature,Correlation
0,2-Year Avg Fires,0.371838
1,slope_mean,0.361036
2,elevation_range,0.340789
3,dominant_province_description_Sierran Steppe-M...,0.323069
4,elevation_std,0.305095
...,...,...
106,dominant_section_description_Mono,0.016665
107,dominant_section_description_Northern Californ...,-0.013201
108,dominant_section_description_Southern Cascades,0.012205
109,power_line_density,0.010760


---

## 3. Export Data

In [13]:
X.to_csv('../data/processed/X.csv', index=False)
y.to_csv('../data/processed/y.csv', index=False)
details.to_csv('../data/processed/details.csv', index=False)

pal_X.to_csv('../data/processed/pal_X.csv', index=False)
pal_y.to_csv('../data/processed/pal_y.csv', index=False)
pal_details.to_csv('../data/processed/pal_details.csv', index=False)

print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/


In [14]:
X

,Burning Index,Energy Release Component,Actual Evapotranspiration,100-hour Dead Fuel Moisture,1000-hour Dead Fuel Moisture,Precipitation,Maximum Relative Humidity,Minimum Relative Humidity,Specific Humidity,Solar Radiation,...,dominant_section_description_Sierra Nevada Foothills,dominant_section_description_Sonoran Desert,dominant_section_description_Southeastern Great Basin,dominant_section_description_Southern California Coast,dominant_section_description_Southern California Mountains and Valleys,dominant_section_description_Southern Cascades,Season_Fall,Season_Spring,Season_Suumer,Season_Winter
0,0.384106,0.666667,0.452282,0.139241,0.135135,0.000000,0.425868,0.116162,0.224469,0.918577,...,False,False,False,False,False,False,False,False,True,False
1,0.344371,0.544715,0.327801,0.224684,0.181081,0.000000,0.791798,0.322222,0.358948,0.889065,...,False,False,False,False,False,False,False,False,True,False
2,0.000000,0.000000,0.045643,0.746835,0.678378,0.019645,0.911672,0.596970,0.124874,0.137022,...,False,False,False,False,False,False,False,False,False,True
3,0.476821,0.837398,0.468880,0.063291,0.067568,0.000000,0.278654,0.069697,0.231547,0.858498,...,False,False,False,False,False,False,False,False,True,False
4,0.198675,0.341463,0.178423,0.310127,0.383784,0.000000,0.677182,0.131313,0.154196,0.461924,...,False,False,False,False,True,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124813,0.245033,0.382114,0.124481,0.417722,0.289189,0.000000,0.702418,0.187879,0.142063,0.311726,...,False,False,False,False,False,False,True,False,False,False
124814,0.185430,0.357724,0.074689,0.455696,0.297297,0.000000,0.992639,0.263636,0.206775,0.262978,...,False,False,False,False,False,False,True,False,False,False
124815,0.172185,0.341463,0.070539,0.484177,0.305405,0.000000,1.000000,0.282828,0.219414,0.346772,...,False,False,False,False,False,False,True,False,False,False
124816,0.238411,0.373984,0.153527,0.436709,0.300000,0.000000,0.779180,0.154545,0.177958,0.335968,...,False,False,False,False,False,False,True,False,False,False
